# llm-finetune-inference-lab on Colab Pro+ (A100 40GB)

Run every cell top to bottom. The notebook is idempotent: small LoRA adapters are kept on Google Drive (so training resumes across sessions), while large merged and quantized weights and all reports are written to local `/content` to avoid the free-tier Drive quota.

If the SFT / DPO adapters already exist on Drive from a previous run, the training cells are skipped automatically and only evaluation and export run.

Set the runtime to **A100 GPU** (Runtime -> Change runtime type) before starting.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

%cd /content
!rm -rf /content/llm-finetune-inference-lab
!git clone https://github.com/dataeclipse/llm-finetune-inference-lab.git
%cd llm-finetune-inference-lab
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"/usr/local/bin:{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
!uv --version
!uv sync --extra train

## Free Drive of large artifacts from previous runs

Removes only the big merged / quantized weights from Drive (they now live on local disk). Small LoRA adapters and preference pairs are kept so training can resume.

In [ ]:
!rm -rf /content/drive/MyDrive/llm-lab/checkpoints/merged
!rm -rf /content/drive/MyDrive/llm-lab/checkpoints/awq
!rm -rf /content/drive/MyDrive/llm-lab/checkpoints/gguf
!du -sh /content/drive/MyDrive/llm-lab/* 2>/dev/null || echo 'nothing on Drive yet'

## Secrets (optional)

Add `WANDB_API_KEY` (and optionally `HF_TOKEN`) in the Colab Secrets panel (key icon, left sidebar) and enable notebook access. Both are optional: without a W&B key, logging is disabled; `HF_TOKEN` is only needed for gated base models (Qwen3-8B is open).

In [ ]:
from google.colab import userdata

try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B logging enabled')
except Exception:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B key not found, logging disabled')

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token set')
except Exception:
    print('HF token not found (fine for Qwen3-8B)')

In [ ]:
!uv run lab data prepare profile=colab_a100
!uv run lab data stats profile=colab_a100

## SFT (QLoRA) - skipped if the adapter already exists on Drive

In [ ]:
SFT_DIR = '/content/drive/MyDrive/llm-lab/checkpoints/sft'
if os.path.exists(f'{SFT_DIR}/adapter_config.json'):
    print('SFT adapter already on Drive, skipping training')
else:
    !uv run lab train sft profile=colab_a100

## Evaluate base model and SFT (execution accuracy on SQLite)

Reports are written to local `/content/work/reports/eval.md`.

In [ ]:
!uv run lab eval run profile=colab_a100 --model-path Qwen/Qwen3-8B
!uv run lab eval run profile=colab_a100 --model-path /content/drive/MyDrive/llm-lab/checkpoints/sft

## DPO - mine preference pairs from SFT execution failures, then train

Skipped entirely if the DPO adapter already exists on Drive. Otherwise: merge the SFT adapter locally, serve it with vLLM, mine pairs, then run DPO.

In [ ]:
import subprocess, time

DPO_DIR = '/content/drive/MyDrive/llm-lab/checkpoints/dpo'
PAIRS = '/content/drive/MyDrive/llm-lab/data/dpo_pairs.jsonl'

if os.path.exists(f'{DPO_DIR}/adapter_config.json'):
    print('DPO adapter already on Drive, skipping pair mining and DPO')
else:
    if not os.path.exists(PAIRS):
        !uv sync --extra train --extra serve
        !uv run lab export merge profile=colab_a100 --adapter-path {SFT_DIR} --output-dir /content/sft-merged
        server = subprocess.Popen(['uv', 'run', 'vllm', 'serve', '/content/sft-merged', '--served-model-name', 'sft', '--max-model-len', '4096'])
        time.sleep(220)
        !uv run lab train pairs profile=colab_a100 --base-url http://localhost:8000/v1 --model sft
        server.terminate()
        time.sleep(10)
        !rm -rf /content/sft-merged
    !uv run lab train dpo profile=colab_a100

## Evaluate DPO and show the before/after table

In [ ]:
!uv run lab eval run profile=colab_a100 --model-path /content/drive/MyDrive/llm-lab/checkpoints/dpo
print('\n===== RESULTS (copy this into the README) =====\n')
!cat /content/work/reports/eval.md

## Merge the final model and export quantized formats (all local)

In [ ]:
!uv sync --extra train --extra quant
!uv run lab export merge profile=colab_a100
!uv run lab export awq profile=colab_a100
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
%env LLAMA_CPP_DIR=/content/llama.cpp
!uv run lab export gguf profile=colab_a100

## Serve the merged model with vLLM and benchmark latency

In [ ]:
!uv sync --extra train --extra serve
server = subprocess.Popen(['uv', 'run', 'lab', 'serve', 'vllm', 'profile=colab_a100'])
time.sleep(220)
!uv run lab bench latency --base-url http://localhost:8000/v1 --concurrency 8 --requests 64
!uv run lab bench gpu --duration 60 --interval 5
print('\n===== serving benchmark =====')
!cat reports/bench.md

## Optional: Gradio chat UI (leave the vLLM server above running)

In [ ]:
!uv sync --extra train --extra serve --extra ui
!uv run python ui/app.py --base-url http://localhost:8000/v1 --model qwen3-8b-sql